# TabICL vs XGBoost Experiment
## Cart-to-Purchase Conversion Prediction

This notebook compares TabICL with XGBoost using a **shared sample** and a **shared train/test split**.

### Why this notebook uses 50K rows instead of 500K
- TabICL is pretrained for roughly 300 to 48K training rows.
- The previous 500K-row version triggered `RuntimeError: Invalid buffer size: 28.61 GB` on Apple Silicon.
- This notebook uses a safer setup for a fair and reproducible comparison on local hardware.

### Key design choices
- One dataset sample is loaded once and reused by both models.
- One train/test split is created once and reused by both models.
- TabICL receives a pandas DataFrame so it can apply its built-in preprocessing.
- XGBoost uses a separate encoded copy of the same split.
- TabICL is forced to CPU with disk offloading for Apple Silicon safety.


## 1. Setup & Imports

In [1]:
import gc
import os
import time
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from sklearn.metrics import accuracy_score, f1_score, log_loss, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split

warnings.filterwarnings('ignore')

if torch.cuda.is_available():
    DEVICE = "cuda"
elif torch.backends.mps.is_available():
    DEVICE = "mps"
else:
    DEVICE = "cpu"

print(f"Detected torch device: {DEVICE}")
print("TabICL will run on CPU to avoid Apple Silicon MPS buffer issues.")


Detected torch device: mps
TabICL will run on CPU to avoid Apple Silicon MPS buffer issues.


## 2. Load Data Once

In [2]:
DATA_PATH = Path("/Users/jky/Library/CloudStorage/GoogleDrive-lethanhquang094@gmail.com/My Drive/FPT/Semester_4/DAP391m/Cart-to-Purchase-Conversion-Prediction-wt-migrate-react/data_pipeline/propensity_feature_store/propensity_features/feature_repo/data/processed_purchase_propensity_data_v2.parquet")
SAMPLE_SIZE = 50_000
TARGET = "is_purchased"

print(f"Loading dataset from: {DATA_PATH}")
df = pd.read_parquet(DATA_PATH)
print(f"Full dataset shape: {df.shape}")

df_sample = df.sample(n=SAMPLE_SIZE, random_state=42).reset_index(drop=True)
del df
gc.collect()

print(f"Sampled dataset shape: {df_sample.shape}")
print(df_sample[TARGET].value_counts(dropna=False))


Loading dataset from: /Users/jky/Library/CloudStorage/GoogleDrive-lethanhquang094@gmail.com/My Drive/FPT/Semester_4/DAP391m/Cart-to-Purchase-Conversion-Prediction-wt-migrate-react/data_pipeline/propensity_feature_store/propensity_features/feature_repo/data/processed_purchase_propensity_data_v2.parquet
Full dataset shape: (2929997, 31)
Sampled dataset shape: (50000, 31)
is_purchased
0    37068
1    12932
Name: count, dtype: int64


## 3. Shared Feature Selection and Shared Split

In [3]:
exclude_cols = [TARGET, "event_time", "user_id", "product_id", "category_code"]
feature_cols = [col for col in df_sample.columns if col not in exclude_cols]
datetime_cols = df_sample[feature_cols].select_dtypes(include=["datetime64", "datetime64", "datetime64"]).columns.tolist()
feature_cols = [col for col in feature_cols if col not in datetime_cols]

X_raw = df_sample[feature_cols].copy()
y = df_sample[TARGET].astype(int).to_numpy()

cat_cols = X_raw.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
num_cols = [col for col in X_raw.columns if col not in cat_cols]

X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print(f"Removed datetime columns: {datetime_cols}")
print(f"Total features: {len(feature_cols)} | categorical: {len(cat_cols)} | numeric: {len(num_cols)}")
print(f"Train shape: {X_train_raw.shape}, Test shape: {X_test_raw.shape}")


Removed datetime columns: ['event_timestamp', 'created_timestamp']
Total features: 26 | categorical: 3 | numeric: 23
Train shape: (40000, 26), Test shape: (10000, 26)


## 4. XGBoost Preparation (Encoded Copy of Shared Split)

In [4]:
def encode_for_xgboost(train_df: pd.DataFrame, test_df: pd.DataFrame, categorical_cols: list[str]):
    train_encoded = train_df.copy()
    test_encoded = test_df.copy()

    for col in categorical_cols:
        categories = pd.Index(train_encoded[col].astype(str).fillna("__missing__").unique())
        mapping = {value: idx for idx, value in enumerate(categories)}

        train_encoded[col] = train_encoded[col].astype(str).fillna("__missing__").map(mapping).astype(np.int32)
        test_encoded[col] = test_encoded[col].astype(str).fillna("__missing__").map(mapping).fillna(-1).astype(np.int32)

    return train_encoded, test_encoded

X_train_xgb, X_test_xgb = encode_for_xgboost(X_train_raw, X_test_raw, cat_cols)
print("XGBoost preprocessing complete.")
print(X_train_xgb.dtypes.head())


XGBoost preprocessing complete.
category_code_level1      int32
category_code_level2      int32
brand                     int32
event_weekday             int64
price                   float64
dtype: object


## 5. Train and Evaluate XGBoost

In [5]:
import xgboost as xgb

xgb_params = {
    "objective": "binary:logistic",
    "eval_metric": ["auc", "logloss"],
    "max_depth": 6,
    "eta": 0.1,
    "subsample": 0.8,
    "colsample_bytree": 0.8,
    "tree_method": "hist",
    "device": "cuda" if DEVICE == "cuda" else "cpu",
}

start_time = time.time()
dtrain = xgb.DMatrix(X_train_xgb, label=y_train)
dtest = xgb.DMatrix(X_test_xgb, label=y_test)
xgb_model = xgb.train(
    xgb_params,
    dtrain,
    num_boost_round=500,
    evals=[(dtrain, "train"), (dtest, "test")],
    early_stopping_rounds=50,
    verbose_eval=50,
)
xgb_train_time = time.time() - start_time

y_prob_xgb = xgb_model.predict(dtest)
y_pred_xgb = (y_prob_xgb >= 0.5).astype(int)

xgb_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_xgb),
    "precision": precision_score(y_test, y_pred_xgb, zero_division=0),
    "recall": recall_score(y_test, y_pred_xgb, zero_division=0),
    "f1": f1_score(y_test, y_pred_xgb, zero_division=0),
    "auc_roc": roc_auc_score(y_test, y_prob_xgb),
    "log_loss": log_loss(y_test, y_prob_xgb),
    "train_time_seconds": xgb_train_time,
}

print("XGBoost results")
for key, value in xgb_metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")


[0]	train-auc:0.90855	train-logloss:0.52955	test-auc:0.90260	test-logloss:0.53017
[50]	train-auc:0.93408	train-logloss:0.27832	test-auc:0.92366	test-logloss:0.29431
[100]	train-auc:0.94626	train-logloss:0.25606	test-auc:0.92470	test-logloss:0.29023
[150]	train-auc:0.95743	train-logloss:0.23627	test-auc:0.92451	test-logloss:0.29024
[163]	train-auc:0.95990	train-logloss:0.23181	test-auc:0.92431	test-logloss:0.29064
XGBoost results
accuracy: 0.8480
precision: 0.7212
recall: 0.6721
f1: 0.6958
auc_roc: 0.9243
log_loss: 0.2906
train_time_seconds: 0.7301


## 6. Train and Evaluate TabICL

In [6]:
from tabicl import TabICLClassifier

del dtrain, dtest
gc.collect()

offload_dir = Path("./tabicl_offload")
offload_dir.mkdir(parents=True, exist_ok=True)

tabicl_model = TabICLClassifier(
    device="cpu",
    n_estimators=4,
    batch_size=2,
    offload_mode="disk",
    disk_offload_dir=str(offload_dir.resolve()),
    kv_cache=False,
    verbose=True,
)

print("TabICL is using the raw DataFrame split with built-in preprocessing.")
start_time = time.time()
tabicl_model.fit(X_train_raw, y_train)
tabicl_train_time = time.time() - start_time

y_prob_tabicl = tabicl_model.predict_proba(X_test_raw)[:, 1]
y_pred_tabicl = (y_prob_tabicl >= 0.5).astype(int)

tabicl_metrics = {
    "accuracy": accuracy_score(y_test, y_pred_tabicl),
    "precision": precision_score(y_test, y_pred_tabicl, zero_division=0),
    "recall": recall_score(y_test, y_pred_tabicl, zero_division=0),
    "f1": f1_score(y_test, y_pred_tabicl, zero_division=0),
    "auc_roc": roc_auc_score(y_test, y_prob_tabicl),
    "log_loss": log_loss(y_test, y_prob_tabicl),
    "train_time_seconds": tabicl_train_time,
}

print("TabICL results")
for key, value in tabicl_metrics.items():
    print(f"{key}: {value:.4f}" if isinstance(value, float) else f"{key}: {value}")


TabICL is using the raw DataFrame split with built-in preprocessing.
Columns classified as categorical: ['category_code_level1', 'category_code_level2', 'brand']
Columns classified as continuous: ['event_weekday', 'price', 'activity_count', 'event_hour', 'user_total_events', 'user_total_views', 'user_total_carts', 'user_total_purchases', 'user_view_to_cart_rate', 'user_cart_to_purchase_rate', 'user_avg_purchase_price', 'user_unique_products', 'user_unique_categories', 'product_total_events', 'product_total_views', 'product_total_carts', 'product_total_purchases', 'product_view_to_cart_rate', 'product_cart_to_purchase_rate', 'product_unique_buyers', 'brand_purchase_rate', 'price_vs_user_avg', 'price_vs_category_avg']
TabICL results
accuracy: 0.8489
precision: 0.7254
recall: 0.6690
f1: 0.6960
auc_roc: 0.9255
log_loss: 0.2884
train_time_seconds: 0.8596


## 7. Comparison Table

In [7]:
comparison_df = pd.DataFrame({
    "Metric": list(xgb_metrics.keys()),
    "XGBoost": [xgb_metrics[k] for k in xgb_metrics.keys()],
    "TabICL": [tabicl_metrics[k] for k in xgb_metrics.keys()],
})
comparison_df["Difference_XGB_minus_TabICL"] = comparison_df["XGBoost"] - comparison_df["TabICL"]
comparison_df


,Metric,XGBoost,TabICL,Difference_XGB_minus_TabICL
0,accuracy,0.848000,0.848900,-0.000900
1,precision,0.721162,0.725367,-0.004205
2,recall,0.672080,0.668987,0.003094
3,f1,0.695757,0.696037,-0.000280
4,auc_roc,0.924292,0.925481,-0.001190
5,log_loss,0.290617,0.288369,0.002248
6,train_time_seconds,0.730108,0.859580,-0.129472


## 8. Conclusion

In [8]:
winner = "XGBoost" if xgb_metrics["auc_roc"] >= tabicl_metrics["auc_roc"] else "TabICL"
print(f"Sample size used for both models: {SAMPLE_SIZE:,}")
print(f"Shared train rows: {len(X_train_raw):,} | Shared test rows: {len(X_test_raw):,}")
print(f"Winner by AUC-ROC: {winner}")
print("If you want a full-scale 500K experiment, keep XGBoost at 500K but run TabICL in a separate reduced-scale benchmark.")


Sample size used for both models: 50,000
Shared train rows: 40,000 | Shared test rows: 10,000
Winner by AUC-ROC: TabICL
If you want a full-scale 500K experiment, keep XGBoost at 500K but run TabICL in a separate reduced-scale benchmark.
